# Stage 01 · Paper Full CUDA Parallel


本 notebook 只负责 Colab 环境准备，并调用 scripts/01_Paper_Full_Cuda_Parallel.py，以两路 prompt-level 并行方式执行 stage 01。



### 初始化运行常量与通用工具
定义仓库路径、Drive 路径、配置文件和脚本入口，并准备命令执行、目录创建、JSON 读取与格式化输出等公共函数。
- 仓库：https://github.com/RICHAAARC/CEG-WM.git
- 分支：organize

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "01_Paper_Full_Cuda_Parallel"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
REPO_ROOT = Path("/content/ceg_wm_workspace")
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_Outputs_Parallel"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
SCRIPT_PATH = REPO_ROOT / "scripts" / "01_Paper_Full_Cuda_Parallel.py"
PARALLEL_WORKER_COUNT = 2
ATTESTATION_ENV_PATH = DRIVE_PROJECT_ROOT / "secrets" / "attestation_env.json"
HF_HOME = REPO_ROOT / "huggingface_cache"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"
PERSISTENT_MODEL_CACHE_ROOT = DRIVE_PROJECT_ROOT / "shared_models"
SEMANTIC_MODEL_SOURCE_URLS = {
    "inspyrenet": "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth",
}

def run_checked(command, cwd=None, env=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)
    return path


def load_json(path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))


### 挂载 Google Drive 并同步代码仓库

挂载 Google Drive、准备项目输出根目录，并克隆或更新 GitHub 仓库，确保后续运行基于正确代码版本。



输出目录根路径是 DRIVE_PROJECT_ROOT，可按下面的树状结构理解：



```text

/content/drive/MyDrive/CEG_WM_Outputs_Parallel/

├─ secrets/

│  └─ attestation_env.json 等运行密钥文件

├─ runtime_state/

│  └─ stage 的运行状态快照与 manifest

├─ exports/ 或脚本内部导出的结果目录

│  └─ 最终可交付产物

└─ logs/ 或各 stage 自带日志目录

   └─ 命令执行日志与调试日志

```


In [ ]:
from google.colab import drive

drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
ensure_dir(DRIVE_PROJECT_ROOT)
ensure_dir(HF_HOME)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_url = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    ).stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

print_json(
    "repo_binding",
    {
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "config_path": str(CONFIG_PATH),
        "script_path": str(SCRIPT_PATH),
    },
)

### 安装依赖、补齐系统环境并准备模型资源
按 pyproject.toml 安装项目最小运行依赖，补充 InSPyReNet 运行包，设置 Hugging Face 缓存，解析配置，下载语义模型权重和基础模型快照，并检查 GPU 是否可用。

绑定两类关键模型路径：
- SD3.5 模型缓存根目录：REPO_ROOT/huggingface_cache
- Hugging Face Hub 缓存目录：REPO_ROOT/huggingface_cache/hub
- Transformers 缓存目录：REPO_ROOT/huggingface_cache/transformers
- SD3.5 模型快照实际落盘路径：MODEL_SNAPSHOT_PATH（由 snapshot_download 返回）
- 语义模型权重路径：REPO_ROOT/<mask.semantic_model_path>
- Drive 持久模型缓存路径：DRIVE_PROJECT_ROOT/shared_models/inspyrenet/ckpt_base.pth
- 当前 notebook 里语义模型来源是 semantic_model_source，对应下载到 InSPyReNet 权重文件

排查模型下载问题时：
- 先看 shared_models 和 WEIGHT_PATH 是否已有可复用权重
- 再看 SD3.5 基础模型是否进入 huggingface_cache 及其 snapshot 目录
- 如果下载完成但运行仍失败，再结合 config_bindings 与 weight_download_summary 检查路径绑定是否一致。

In [ ]:
import urllib.request

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if Path("/usr/bin/apt-get").exists():
    subprocess.run(["apt-get", "update", "-qq"], check=False, capture_output=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"],
        check=False,
        capture_output=True,
    )

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)

if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

run_checked([sys.executable, "-m", "pip", "install", "transparent-background"], cwd=REPO_ROOT)

from huggingface_hub import HfApi, snapshot_download
from scripts.notebook_runtime_common import (
    build_directory_digest_summary,
    collect_weight_summary,
    load_yaml_mapping,
    resolve_model_identity,
)

CFG_OBJ = load_yaml_mapping(CONFIG_PATH)
MODEL_IDENTITY = resolve_model_identity(CFG_OBJ)
MASK_CFG = CFG_OBJ.get("mask", {}) if isinstance(CFG_OBJ.get("mask"), dict) else {}
PROMPT_SOURCE_PATH = REPO_ROOT / str(CFG_OBJ.get("inference_prompt_file", ""))
WEIGHT_PATH = REPO_ROOT / str(MASK_CFG.get("semantic_model_path", ""))
WEIGHT_SOURCE = str(MASK_CFG.get("semantic_model_source", ""))
WEIGHT_URL = str(os.environ.get("INSPYRENET_MODEL_URL", SEMANTIC_MODEL_SOURCE_URLS.get(WEIGHT_SOURCE, ""))).strip()
PERSISTENT_WEIGHT_PATH = PERSISTENT_MODEL_CACHE_ROOT / "inspyrenet" / "ckpt_base.pth"
FORCE_WEIGHT_DOWNLOAD = str(os.environ.get("INSPYRENET_FORCE_DOWNLOAD", "")).strip().lower() in {"1", "true", "yes"}
ensure_dir(WEIGHT_PATH.parent)
ensure_dir(PERSISTENT_WEIGHT_PATH.parent)

def is_valid_weight_file(path_obj):
    return path_obj.exists() and path_obj.is_file() and path_obj.stat().st_size > 0

def sync_weight_to_repo(source_path, target_path):
    if source_path.resolve() == target_path.resolve():
        return
    ensure_dir(target_path.parent)
    shutil.copy2(source_path, target_path)

HF_ACCESS_SUMMARY = {"status": "unknown", "detail": "not_checked"}
try:
    hf_user = HfApi().whoami()
    HF_ACCESS_SUMMARY = {"status": "authenticated", "detail": hf_user.get("name", "unknown")}
except Exception:
    HF_ACCESS_SUMMARY = {"status": "anonymous", "detail": "anonymous access"}

existing_weight_path = None
for candidate_path in [WEIGHT_PATH, PERSISTENT_WEIGHT_PATH]:
    if is_valid_weight_file(candidate_path):
        existing_weight_path = candidate_path
        break

if existing_weight_path is not None and not FORCE_WEIGHT_DOWNLOAD:
    sync_weight_to_repo(existing_weight_path, WEIGHT_PATH)
    if existing_weight_path.resolve() != PERSISTENT_WEIGHT_PATH.resolve():
        shutil.copy2(existing_weight_path, PERSISTENT_WEIGHT_PATH)
else:
    if not WEIGHT_URL:
        raise RuntimeError(f"unsupported semantic_model_source for notebook bootstrap: {WEIGHT_SOURCE}")
    temp_download_path = PERSISTENT_WEIGHT_PATH.with_suffix(PERSISTENT_WEIGHT_PATH.suffix + ".downloading")
    if temp_download_path.exists():
        temp_download_path.unlink()
    urllib.request.urlretrieve(WEIGHT_URL, temp_download_path)
    if not is_valid_weight_file(temp_download_path):
        temp_download_path.unlink(missing_ok=True)
        raise RuntimeError(f"downloaded semantic weight is invalid: {temp_download_path}")
    temp_download_path.replace(PERSISTENT_WEIGHT_PATH)
    sync_weight_to_repo(PERSISTENT_WEIGHT_PATH, WEIGHT_PATH)

MODEL_SNAPSHOT_PATH = snapshot_download(
    repo_id=str(MODEL_IDENTITY["model_id"]),
    revision=None if MODEL_IDENTITY["revision"] == "<absent>" else str(MODEL_IDENTITY["revision"]),
    cache_dir=str(HF_HOME),
)
MODEL_DOWNLOAD_SUMMARY = build_directory_digest_summary(Path(MODEL_SNAPSHOT_PATH))
WEIGHT_DOWNLOAD_SUMMARY = collect_weight_summary(REPO_ROOT, CFG_OBJ)
CONFIG_BINDINGS = {
    "prompt_source_path": str(PROMPT_SOURCE_PATH),
    "prompt_exists": PROMPT_SOURCE_PATH.exists(),
    "model_id": MODEL_IDENTITY["model_id"],
    "hf_revision": MODEL_IDENTITY["revision"],
    "hf_access": HF_ACCESS_SUMMARY,
    "model_snapshot_path": MODEL_SNAPSHOT_PATH,
    "semantic_model_path": str(WEIGHT_PATH),
    "semantic_model_cache_path": str(PERSISTENT_WEIGHT_PATH),
    "semantic_model_source": WEIGHT_SOURCE,
    "semantic_model_url": WEIGHT_URL,
    "force_weight_download": FORCE_WEIGHT_DOWNLOAD,
}
print_json("config_bindings", CONFIG_BINDINGS)
print_json("model_download_summary", MODEL_DOWNLOAD_SUMMARY)
print_json("weight_download_summary", WEIGHT_DOWNLOAD_SUMMARY)
run_checked(["nvidia-smi"])

In [ ]:
from scripts.notebook_runtime_common import ensure_attestation_env_bootstrap

ATTESTATION_BOOTSTRAP = ensure_attestation_env_bootstrap(
    CFG_OBJ,
    DRIVE_PROJECT_ROOT,
    allow_generate=True,
    allow_missing=False,
 )
print_json("attestation_env_bootstrap", ATTESTATION_BOOTSTRAP)

### 环境预检与运行前门禁
在执行主流程脚本前，先检查配置文件、脚本文件、关键模块、HF 模型可访问性、InSPyReNet 权重状态与磁盘空间。若存在硬性失败项，应先修复再继续执行。

In [ ]:
import os
import socket

from scripts.notebook_runtime_common import (
    apply_notebook_model_snapshot_binding,
    load_yaml_mapping,
    write_yaml_mapping,
)
from scripts.workflow_acceptance_common import detect_stage_01_preflight

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck("原始配置路径", CONFIG_PATH.exists(), str(CONFIG_PATH))
record_precheck("主流程脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("prompt 文件存在", PROMPT_SOURCE_PATH.exists(), str(PROMPT_SOURCE_PATH))
record_precheck(
    "MODEL_SNAPSHOT_PATH 已提供",
    isinstance(MODEL_SNAPSHOT_PATH, str) and bool(str(MODEL_SNAPSHOT_PATH).strip()),
    str(MODEL_SNAPSHOT_PATH),
)

PRECHECK_RUNTIME_METADATA_ROOT = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / "precheck_runtime_metadata"
PRECHECK_BOUND_CONFIG_PATH = PRECHECK_RUNTIME_METADATA_ROOT / "precheck_bound_config.yaml"
PRECHECK_BINDING_ENV = dict(os.environ)
PRECHECK_BINDING_ENV["CEG_WM_MODEL_SNAPSHOT_PATH"] = str(MODEL_SNAPSHOT_PATH)
PRECHECK_BASE_CFG = load_yaml_mapping(CONFIG_PATH)
PRECHECK_BOUND_CFG = apply_notebook_model_snapshot_binding(
    PRECHECK_BASE_CFG,
    env_mapping=PRECHECK_BINDING_ENV,
 )
write_yaml_mapping(PRECHECK_BOUND_CONFIG_PATH, PRECHECK_BOUND_CFG)
PRECHECK_MODEL_SOURCE_BINDING = (
    PRECHECK_BOUND_CFG.get("model_source_binding")
    if isinstance(PRECHECK_BOUND_CFG.get("model_source_binding"), dict)
    else None
)
PRECHECK_BINDING_APPLIED = (
    PRECHECK_MODEL_SOURCE_BINDING is not None
    and isinstance(PRECHECK_BOUND_CFG.get("model_snapshot_path"), str)
    and bool(str(PRECHECK_BOUND_CFG.get("model_snapshot_path")).strip())
)

record_precheck(
    "precheck 绑定配置路径",
    PRECHECK_BOUND_CONFIG_PATH.exists(),
    str(PRECHECK_BOUND_CONFIG_PATH),
)
record_precheck(
    "notebook 模型绑定已应用到 precheck 配置",
    PRECHECK_BINDING_APPLIED,
    json.dumps(PRECHECK_MODEL_SOURCE_BINDING, ensure_ascii=False, sort_keys=True)
    if PRECHECK_MODEL_SOURCE_BINDING is not None
    else "<absent>",
)

STAGE_01_PREFLIGHT = detect_stage_01_preflight(PRECHECK_BOUND_CONFIG_PATH)
record_precheck(
    "stage 01 preflight",
    STAGE_01_PREFLIGHT["ok"],
    json.dumps(STAGE_01_PREFLIGHT, ensure_ascii=False, sort_keys=True),
)
record_precheck(
    "CUDA 可用",
    STAGE_01_PREFLIGHT["gpu_tool_available"],
    STAGE_01_PREFLIGHT["nvidia_smi_path"],
)
record_precheck(
    "attestation env 已就绪",
    not STAGE_01_PREFLIGHT["missing_attestation_env_vars"],
    ", ".join(STAGE_01_PREFLIGHT["missing_attestation_env_vars"]) or "ok",
)
record_precheck(
    "InSPyReNet 权重存在",
    is_valid_weight_file(WEIGHT_PATH),
    f"path={WEIGHT_PATH}, cache_path={PERSISTENT_WEIGHT_PATH}",
)

for module_name in ["yaml", "huggingface_hub", "diffusers", "transformers", "safetensors"]:
    try:
        __import__(module_name)
        record_precheck(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        record_precheck(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

hf_model_ok = False
hf_model_detail = "not_checked"
try:
    socket.setdefaulttimeout(15)
    model_info = HfApi().model_info(str(MODEL_IDENTITY["model_id"]))
    hf_model_ok = True
    hf_model_detail = f"accessible: {model_info.id}"
except Exception as exc:
    hf_model_ok = False
    hf_model_detail = f"{type(exc).__name__}: {exc}"
record_precheck("HF 模型可访问", hf_model_ok, hf_model_detail)

disk_usage = shutil.disk_usage(str(REPO_ROOT))
free_gb = disk_usage.free / (1024 ** 3)
record_precheck("磁盘剩余空间", free_gb >= 20.0, f"free={free_gb:.1f} GB，建议 >= 20 GB")

hard_fail = []
if not STAGE_01_PREFLIGHT["ok"]:
    hard_fail.append({
        "name": "stage 01 preflight",
        "detail": json.dumps(STAGE_01_PREFLIGHT, ensure_ascii=False, sort_keys=True),
    })

print_json(
    "environment_precheck",
    {
        "config_path": str(CONFIG_PATH),
        "precheck_bound_config_path": str(PRECHECK_BOUND_CONFIG_PATH),
        "model_snapshot_path": str(MODEL_SNAPSHOT_PATH),
        "notebook_model_snapshot_binding_applied": PRECHECK_BINDING_APPLIED,
        "precheck_bound_config": {
            "model_snapshot_path": PRECHECK_BOUND_CFG.get("model_snapshot_path", "<absent>"),
            "model_source_binding": PRECHECK_MODEL_SOURCE_BINDING or "<absent>",
        },
        "items": PRECHECK_RESULTS,
        "stage_01_preflight": STAGE_01_PREFLIGHT,
        "hard_fail": hard_fail,
    },
)

if hard_fail:
    raise RuntimeError(f"environment precheck failed: {hard_fail}")

### 执行主流程脚本

生成本次运行的 stage_run_id，调用 scripts/01_Paper_Full_Cuda_Parallel.py，并以固定 2 个 worker 执行 prompt-level 并行 source pool，脚本结束后再读取 stage summary 与相关 manifest。



这一阶段最关键的输出目录结构可以理解为：



```text

DRIVE_PROJECT_ROOT/

└─ runtime_state/

   └─ 01_Paper_Full_Cuda_Parallel/

      └─ <stage_run_id>/

         └─ stage_summary.json

            ├─ stage_manifest_path

            ├─ package_manifest_path

            ├─ run_root

            ├─ log_root

            └─ export_root

```



也就是说，后续校验和诊断并不是重新猜路径，而是统一从 stage_summary.json 反查所有输出位置。


In [ ]:
from pathlib import Path



from scripts.notebook_runtime_common import build_repo_import_subprocess_env, make_stage_run_id, resolve_stage_roots





def read_log_tail(path_obj, max_lines=40):

    if max_lines <= 0:

        raise ValueError("max_lines must be positive int")

    path_obj = Path(path_obj)

    if not path_obj.exists() or not path_obj.is_file():

        return []

    return path_obj.read_text(encoding="utf-8", errors="replace").splitlines()[-max_lines:]





def load_json_if_exists(path_obj):

    path_obj = Path(path_obj)

    if not path_obj.exists() or not path_obj.is_file():

        return None

    with path_obj.open("r", encoding="utf-8") as handle:

        return json.load(handle)





def collect_recent_logs(log_paths, max_logs=8, max_lines=40):

    unique_paths = {}

    for path_obj in log_paths:

        path_obj = Path(path_obj)

        if path_obj.exists() and path_obj.is_file():

            unique_paths[str(path_obj)] = path_obj

    sorted_paths = sorted(

        unique_paths.values(),

        key=lambda item: item.stat().st_mtime if item.exists() else 0,

        reverse=True,

    )

    return [

        {

            "path": str(path_obj),

            "tail": read_log_tail(path_obj, max_lines=max_lines),

        }

        for path_obj in sorted_paths[:max_logs]

    ]





STAGE_RUN_ID = make_stage_run_id(NOTEBOOK_NAME)

STAGE_ROOTS = resolve_stage_roots(DRIVE_PROJECT_ROOT, NOTEBOOK_NAME, STAGE_RUN_ID)

SUMMARY_PATH = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / STAGE_RUN_ID / "stage_summary.json"

COMMAND = [

    sys.executable,

    str(SCRIPT_PATH),

    "--config",

    str(CONFIG_PATH),

    "--drive-project-root",

    str(DRIVE_PROJECT_ROOT),

    "--stage-run-id",

    STAGE_RUN_ID,

    "--worker-count",

    str(PARALLEL_WORKER_COUNT),

]

NOTEBOOK_SUBPROCESS_ENV = build_repo_import_subprocess_env(repo_root=REPO_ROOT)

NOTEBOOK_SUBPROCESS_ENV["CEG_WM_MODEL_SNAPSHOT_PATH"] = str(MODEL_SNAPSHOT_PATH)



command_result = subprocess.run(

    COMMAND,

    cwd=REPO_ROOT,

    env=NOTEBOOK_SUBPROCESS_ENV,

    check=False,

    capture_output=True,

    text=True,

    encoding="utf-8",

    errors="replace",

)



if command_result.returncode != 0:

    main_stdout_log = STAGE_ROOTS["log_root"] / "01_mainline_stdout.log"

    main_stderr_log = STAGE_ROOTS["log_root"] / "01_mainline_stderr.log"

    workflow_summary_path = STAGE_ROOTS["run_root"] / "artifacts" / "workflow_summary.json"

    stage_summary = load_json_if_exists(SUMMARY_PATH)

    audit_summary_path = (

        Path(stage_summary["audit_summary_path"])

        if isinstance(stage_summary, dict)

        and isinstance(stage_summary.get("audit_summary_path"), str)

        and stage_summary["audit_summary_path"] != "<absent>"

        else None

    )

    nested_log_candidates = []

    for candidate_root in [

        STAGE_ROOTS["log_root"],

        STAGE_ROOTS["run_root"] / "logs",

        STAGE_ROOTS["run_root"] / "source_pool",

    ]:

        candidate_root = Path(candidate_root)

        if candidate_root.exists():

            nested_log_candidates.extend(candidate_root.rglob("*.log"))

    failure_diagnostics = {

        "stage_run_id": STAGE_RUN_ID,

        "parallel_worker_count": PARALLEL_WORKER_COUNT,

        "return_code": int(command_result.returncode),

        "command_stdout_tail": command_result.stdout.splitlines()[-40:],

        "command_stderr_tail": command_result.stderr.splitlines()[-40:],

        "log_root": str(STAGE_ROOTS["log_root"]),

        "run_root": str(STAGE_ROOTS["run_root"]),

        "runtime_state_root": str(STAGE_ROOTS["runtime_state_root"]),

        "export_root": str(STAGE_ROOTS["export_root"]),

        "main_stdout_log": str(main_stdout_log),

        "main_stderr_log": str(main_stderr_log),

        "main_stdout_tail": read_log_tail(main_stdout_log),

        "main_stderr_tail": read_log_tail(main_stderr_log),

        "workflow_summary_path": str(workflow_summary_path),

        "workflow_summary": load_json_if_exists(workflow_summary_path),

        "stage_summary_path": str(SUMMARY_PATH),

        "stage_summary": stage_summary,

        "audit_summary_path": str(audit_summary_path) if audit_summary_path else "<absent>",

        "audit_summary": load_json_if_exists(audit_summary_path) if audit_summary_path else None,

        "recent_nested_logs": collect_recent_logs(nested_log_candidates),

    }

    print_json("stage_01_failure_diagnostics", failure_diagnostics)

    raise RuntimeError(

        "stage 01 notebook execution failed; inspect stage_01_failure_diagnostics for workflow summary, stage summary, audit summary, and log tails"

    )



STAGE_SUMMARY = load_json(SUMMARY_PATH)

STAGE_MANIFEST = load_json(Path(STAGE_SUMMARY["stage_manifest_path"]))

PACKAGE_MANIFEST = load_json(Path(STAGE_SUMMARY["package_manifest_path"]))

AUDIT_SUMMARY_PATH = Path(STAGE_SUMMARY["audit_summary_path"])

AUDIT_SUMMARY = load_json(AUDIT_SUMMARY_PATH)

print_json("stage_summary", STAGE_SUMMARY)

print_json("stage_manifest", STAGE_MANIFEST)

print_json("package_manifest", PACKAGE_MANIFEST)

print_json("audit_summary", AUDIT_SUMMARY)



if STAGE_SUMMARY.get("audit_status") != "passed":

    raise RuntimeError(

        f"stage 01 audit blocked downstream stages: {AUDIT_SUMMARY.get('blocking_reasons', [])}"

    )


### 校验关键产物完整性
检查 formal stage outputs、canonical source pool 相关合同与打包文件是否齐全，并将 root embed/detect records 作为 optional compatibility views 单独汇总。

重点校验的是 run_root 下的 formal outputs 是否完整，通常可以按下面的树状路径理解：

```text
<run_root>/
├─ records/
│  ├─ embed_record.json                     [compatibility view，可选]
│  ├─ detect_record.json                    [compatibility view，可选]
│  ├─ calibration_record.json               [formal required]
│  └─ evaluate_record.json                  [formal required]
├─ artifacts/
│  ├─ thresholds/
│  │  ├─ thresholds_artifact.json           [formal required]
│  │  └─ threshold_metadata_artifact.json   [formal required]
│  ├─ stage_01_canonical_source_pool/
│  │  └─ source_pool_manifest.json          [formal required]
│  ├─ parallel_attestation_statistics_input_contract.json
│  │                                         [formal required]
│  ├─ stage_01_pooled_threshold_build_contract.json
│  │                                         [formal required]
│  ├─ evaluation_report.json                [formal required]
│  ├─ workflow_summary.json                 [formal required]
│  └─ run_closure.json                      [formal required]
├─ stage_manifest.json 或 stage_manifest_path 指向的文件
└─ package_manifest.json 或 package_manifest_path 指向的文件
```

用于确认 canonical source pool 及其 formal outputs 仍是 authoritative source truth；root embed/detect records 只是 compatibility views。若这两项缺失，只记录为 optional missing，不阻断 notebook 校验。

In [ ]:
from scripts.notebook_runtime_common import collect_missing_file_entries, compute_file_sha256

PACKAGE_PATH = Path(STAGE_SUMMARY["package_path"])

REQUIRED_FORMAL_FILES = {
    "stage_manifest": Path(STAGE_SUMMARY["stage_manifest_path"]),
    "package_manifest": Path(STAGE_SUMMARY["package_manifest_path"]),
    "package_zip": PACKAGE_PATH,
    "audit_summary": AUDIT_SUMMARY_PATH,
    "calibration_record": Path(STAGE_MANIFEST["records"]["calibration_record"]["path"]),
    "evaluate_record": Path(STAGE_MANIFEST["records"]["evaluate_record"]["path"]),
    "thresholds_artifact": Path(STAGE_MANIFEST["thresholds_path"]),
    "threshold_metadata_artifact": Path(STAGE_MANIFEST["threshold_metadata_artifact_path"]),
    "canonical_source_pool_manifest": Path(STAGE_MANIFEST["stage_01_canonical_source_pool_manifest_path"]),
    "source_contract": Path(STAGE_MANIFEST["parallel_attestation_statistics_input_contract_path"]),
    "pooled_threshold_build_contract": Path(STAGE_MANIFEST["stage_01_pooled_threshold_build_contract_path"]),
    "evaluation_report": Path(STAGE_MANIFEST["evaluation_report_path"]),
    "run_closure": Path(STAGE_MANIFEST["run_closure_path"]),
    "workflow_summary": Path(STAGE_MANIFEST["workflow_summary_path"]),
    "runtime_config_snapshot": Path(STAGE_MANIFEST["runtime_config_snapshot_path"]),
    "prompt_snapshot": Path(STAGE_MANIFEST["prompt_snapshot_path"]),
}
OPTIONAL_COMPATIBILITY_FILES = {
    "embed_record": Path(STAGE_MANIFEST["records"]["embed_record"]["path"]),
    "detect_record": Path(STAGE_MANIFEST["records"]["detect_record"]["path"]),
}
MISSING_REQUIRED_FORMAL_FILES = collect_missing_file_entries(REQUIRED_FORMAL_FILES)
MISSING_OPTIONAL_COMPATIBILITY_FILES = collect_missing_file_entries(OPTIONAL_COMPATIBILITY_FILES)
OPTIONAL_COMPATIBILITY_STATUS = {
    key: {
        "path": str(value),
        "exists": bool(value.exists() and value.is_file()),
    }
    for key, value in OPTIONAL_COMPATIBILITY_FILES.items()
}
PACKAGE_SHA256 = compute_file_sha256(PACKAGE_PATH)
if PACKAGE_SHA256 != PACKAGE_MANIFEST["package_sha256"]:
    raise RuntimeError(
        f"package sha256 mismatch: manifest={PACKAGE_MANIFEST['package_sha256']} actual={PACKAGE_SHA256}"
    )
if not str(STAGE_MANIFEST["config_source_path"]).endswith("/configs/default.yaml"):
    raise RuntimeError(f"config_source_path is not bound to default.yaml: {STAGE_MANIFEST['config_source_path']}")

print_json(
    "required_formal_files_check",
    {
        "source_truth": STAGE_MANIFEST.get("stage_01_source_truth", "canonical_source_pool"),
        "checked_paths": {key: str(value) for key, value in REQUIRED_FORMAL_FILES.items()},
        "missing_files": MISSING_REQUIRED_FORMAL_FILES,
    },
)
print_json(
    "optional_compatibility_views_check",
    {
        "source_truth": STAGE_MANIFEST.get("stage_01_source_truth", "canonical_source_pool"),
        "view_role": "compatibility_view",
        "checked_paths": OPTIONAL_COMPATIBILITY_STATUS,
        "missing_files": MISSING_OPTIONAL_COMPATIBILITY_FILES,
        "missing_policy": "record_optional_missing_without_failing_notebook_validation",
    },
)

if MISSING_REQUIRED_FORMAL_FILES:
    raise FileNotFoundError(
        f"stage 01 validation failed, missing required formal files: {MISSING_REQUIRED_FORMAL_FILES}"
    )

VALIDATION_RESULT = {
    "stage_name": NOTEBOOK_NAME,
    "stage_run_id": STAGE_RUN_ID,
    "package_sha256": PACKAGE_SHA256,
    "audit_status": STAGE_SUMMARY["audit_status"],
    "source_truth": STAGE_MANIFEST.get("stage_01_source_truth", "canonical_source_pool"),
    "required_formal_files": {key: str(value) for key, value in REQUIRED_FORMAL_FILES.items()},
    "optional_compatibility_views": OPTIONAL_COMPATIBILITY_STATUS,
    "missing_required_formal_files": MISSING_REQUIRED_FORMAL_FILES,
    "missing_optional_compatibility_files": MISSING_OPTIONAL_COMPATIBILITY_FILES,
    "status": "ok",
}
print_json("validation_result", VALIDATION_RESULT)

### 输出诊断信息与日志摘要
汇总运行目录、日志目录、最近日志内容和关键 manifest 字段，便于在流程完成后快速定位问题。

重点回看以下几个输出目录：
- run_root：本次主运行目录，通常承载 records、artifacts、reports 等实际产物
- log_root：日志目录，会递归搜索其中的 .log 文件并提取最新日志尾部
- runtime_state_root：保存 notebook 级别的 stage 运行状态索引
- export_root：保存最终导出或可交付结果

如果前面某一步失败，最先应检查 log_root；如果想看本次运行生成了哪些正式产物，应优先检查 run_root 和 export_root。

In [ ]:
from scripts.notebook_runtime_common import summarize_manifest_fields, tail_text_file

LOG_ROOT = Path(STAGE_SUMMARY["log_root"])

LOG_FILES = sorted(LOG_ROOT.rglob("*.log"), key=lambda item: item.stat().st_mtime if item.exists() else 0)
LATEST_LOG_PATH = LOG_FILES[-1] if LOG_FILES else None
MISSING_REQUIRED_FORMAL_FILES = list(VALIDATION_RESULT.get("missing_required_formal_files", []))
MISSING_OPTIONAL_COMPATIBILITY_FILES = list(VALIDATION_RESULT.get("missing_optional_compatibility_files", []))
if MISSING_REQUIRED_FORMAL_FILES:
    DIAGNOSTIC_STATUS = "validation_incomplete"
elif MISSING_OPTIONAL_COMPATIBILITY_FILES:
    DIAGNOSTIC_STATUS = "optional_compatibility_missing"
else:
    DIAGNOSTIC_STATUS = "ok"

DIAGNOSTIC_RESULT = {
    "stage_run_id": STAGE_RUN_ID,
    "run_root": STAGE_SUMMARY["run_root"],
    "log_root": STAGE_SUMMARY["log_root"],
    "runtime_state_root": STAGE_SUMMARY["runtime_state_root"],
    "export_root": STAGE_SUMMARY["export_root"],
    "missing_required_formal_files": MISSING_REQUIRED_FORMAL_FILES,
    "missing_optional_compatibility_files": MISSING_OPTIONAL_COMPATIBILITY_FILES,
    "latest_log_path": str(LATEST_LOG_PATH) if LATEST_LOG_PATH else "<absent>",
    "latest_log_tail": tail_text_file(LATEST_LOG_PATH, max_lines=20) if LATEST_LOG_PATH else [],
    "stage_manifest_summary": summarize_manifest_fields(
        STAGE_MANIFEST,
        [
            "stage_name",
            "stage_run_id",
            "config_source_path",
            "prompt_snapshot_path",
            "runtime_config_snapshot_path",
            "thresholds_path",
            "workflow_summary_path",
        ],
    ),
    "package_manifest_summary": summarize_manifest_fields(
        PACKAGE_MANIFEST,
        [
            "package_filename",
            "package_sha256",
            "package_created_at",
            "package_contents_index_path",
            "package_manifest_version",
        ],
    ),
    "audit_summary": summarize_manifest_fields(
        AUDIT_SUMMARY,
        [
            "overall_status",
            "definition_status",
            "strong_compatibility_status",
            "stage_02_ready",
            "stage_03_ready",
            "stage_04_ready",
        ],
    ),
    "status": DIAGNOSTIC_STATUS,
}
print_json("diagnostic_result", DIAGNOSTIC_RESULT)

### 最终保存在 Google Drive 中的文件路径

本 notebook 的最终产物都会保存在 Google Drive 挂载目录下，建议按下面的目录树查看：



```text

/content/drive/MyDrive/CEG_WM_Outputs_Parallel/

├─ runtime_state/

│  └─ 01_Paper_Full_Cuda_Parallel/

│     └─ <stage_run_id>/

│        └─ stage_summary.json

├─ <run_root，对应 stage_summary.json 中的 run_root>/

│  ├─ records/

│  │  ├─ embed_record.json

│  │  ├─ detect_record.json

│  │  ├─ calibration_record.json

│  │  └─ evaluate_record.json

│  ├─ artifacts/

│  │  ├─ stage_01_parallel_workers/

│  │  │  ├─ shard_plan.json

│  │  │  └─ worker_XX_result.json

│  │  ├─ thresholds/

│  │  │  ├─ thresholds_artifact.json

│  │  │  └─ threshold_metadata_artifact.json

│  │  ├─ evaluation_report.json

│  │  ├─ workflow_summary.json

│  │  ├─ run_closure.json

│  │  └─ stage_manifest.json

│  └─ logs/ 或脚本生成的日志目录

│     └─ *.log

├─ <export_root，对应 stage_summary.json 中的 export_root>/

│  ├─ 01_Paper_Full_Cuda_Parallel__<stage_run_id>.zip

│  ├─ package_manifest.json

│  └─ package_index.json

└─ <package_path，对应 stage_summary.json 中的 package_path>

   └─ 最终 zip 包

```



最终 zip 包只包含 stage 01 自己的产物，不包含任何上游 stage 内容。zip 内部的核心相对路径如下：



```text

records/embed_record.json

records/detect_record.json

records/calibration_record.json

records/evaluate_record.json

artifacts/stage_01_parallel_workers/shard_plan.json

artifacts/thresholds/thresholds_artifact.json

artifacts/thresholds/threshold_metadata_artifact.json

artifacts/evaluation_report.json

artifacts/run_closure.json

artifacts/workflow_summary.json

artifacts/stage_manifest.json

artifacts/package_manifest.json

artifacts/package_index.json

runtime_metadata/runtime_config_snapshot.yaml

```



建议的查找顺序是：

1. 先打开 stage_summary.json。

2. 再根据其中的 run_root、log_root、export_root、package_path 跳转到真实落盘位置。

3. 如果只关心最终交付结果，优先看 package_path 和 export_root。

4. 如果要排查运行问题，优先看 log_root、run_root，以及 artifacts/stage_01_parallel_workers 下的 worker 结果与日志。
